# Notebook 08 — Customer-Specific Product Filtering

In [ ]:
from nbe_report_functions_final_fixed_v3_reviewed import (
    build_standard_scenarios,
    notebook_output_dir,
    build_customer_product_set,
    mask_wide_table_by_customer_product_set,
    compute_customer_specific_product_count_distribution,
    compute_topk_eligibility_summary,
    export_customer_specific_filter_outputs,
    run_nbe_scenario_outputs,
    compute_changed_product_composition_analysis,
)

BASE_OUTPUT_ROOT = "/dbfs/FileStore/nbe_reports_rebuilt"
CUSTOMER_ID_COL = "customer_id"
CONTROL_COL = "control"
TOP_K_VALUES = [1, 3, 5]

ALL_37_PRODUCTS = [
    "product_1", "product_2", "product_3", "product_4", "product_5",
    "product_6", "product_7", "product_8", "product_9", "product_10",
    "product_11", "product_12", "product_13", "product_14", "product_15",
    "product_16", "product_17", "product_18", "product_19", "product_20",
    "product_21", "product_22", "product_23", "product_24", "product_25",
    "product_26", "product_27", "product_28", "product_29", "product_30",
    "product_31", "product_32", "product_33", "product_34", "product_35",
    "product_36", "product_37",
]

CHANGED_8_PRODUCTS = [
    "product_1", "product_2", "product_3", "product_4",
    "product_5", "product_6", "product_7", "product_8"
]

ACTIVE_12_PRODUCTS = [
    "product_1", "product_2", "product_3", "product_4",
    "product_5", "product_6", "product_7", "product_8",
    "product_9", "product_10", "product_11", "product_12"
]

ACTIVE_CHANGED_PRODUCTS = [
    "product_1", "product_2", "product_3", "product_4"
]


NOTEBOOK_NAME = "notebook_08_customer_specific_product_filtering"


In [ ]:
# legacy_df = spark.table("your_catalog.your_schema.legacy_priority_table_nov")
# new_df = spark.table("your_catalog.your_schema.prism_priority_table_nov")
# filter_long_df = spark.table("your_catalog.your_schema.customer_product_filter_long")  # customer_id, product
# one_million_customer_df = spark.table("your_catalog.your_schema.customer_subset_1m").select(CUSTOMER_ID_COL).distinct()

legacy_df = None
new_df = None
filter_long_df = None
one_million_customer_df = None

if legacy_df is None or new_df is None or filter_long_df is None:
    raise ValueError("Please load legacy_df, new_df, and filter_long_df before running.")

SCENARIOS = build_standard_scenarios(ALL_37_PRODUCTS, CHANGED_8_PRODUCTS, ACTIVE_12_PRODUCTS, ACTIVE_CHANGED_PRODUCTS, one_million_customer_df, TOP_K_VALUES)
scenario = SCENARIOS[2]  # active 12 by default
output_dir = notebook_output_dir(BASE_OUTPUT_ROOT, scenario["scenario_name"], NOTEBOOK_NAME)

customer_product_set_df = build_customer_product_set(filter_long_df, CUSTOMER_ID_COL, "product")
masked_legacy = mask_wide_table_by_customer_product_set(legacy_df, customer_product_set_df, scenario["product_cols"], CUSTOMER_ID_COL)
masked_prism = mask_wide_table_by_customer_product_set(new_df, customer_product_set_df, scenario["product_cols"], CUSTOMER_ID_COL)

count_report = compute_customer_specific_product_count_distribution(masked_legacy, scenario["product_cols"], CUSTOMER_ID_COL)
eligibility_df = compute_topk_eligibility_summary(count_report["customer_product_counts"], TOP_K_VALUES)
artifacts = export_customer_specific_filter_outputs(count_report["distribution"], eligibility_df, output_dir)
print(artifacts)
display(count_report["distribution"])
display(eligibility_df)

# Then rerun the full polished comparison on masked tables
scenario_root = output_dir.parent
core_out = run_nbe_scenario_outputs(
    spark=spark,
    legacy_df=masked_legacy,
    new_df=masked_prism,
    scenario_name=scenario["scenario_name"] + "_customer_specific",
    base_output_path=scenario_root,
    customer_subset=scenario["customer_subset"],
    product_cols=scenario["product_cols"],
    top_k_values=scenario["top_k_values"],
    customer_id_col=CUSTOMER_ID_COL,
    control_col="control",
    export_excel=True,
)
print(core_out)


# New granular changed-product composition analysis
composition_dir = output_dir / '05_changed_product_composition_analysis'
composition_result = compute_changed_product_composition_analysis(
    legacy_df=legacy_df,
    prism_df=new_df,
    customer_filter_long_df=filter_long_df,
    active_product_cols=scenario["product_cols"],
    changed_product_cols=ACTIVE_CHANGED_PRODUCTS if set(scenario["product_cols"]).issuperset(set(ACTIVE_CHANGED_PRODUCTS)) else [p for p in ACTIVE_CHANGED_PRODUCTS if p in scenario["product_cols"]],
    output_dir=composition_dir,
    customer_id_col=CUSTOMER_ID_COL,
    product_col='product',
    control_col=CONTROL_COL,
    filter_control_zero=True,
)
print('Composition artifacts:', composition_result['artifact_paths'])
display(composition_result['overall_summary_tables']['overall_k_summary'])
display(composition_result['overall_summary_tables']['changed_count_mix_by_k'])
# Example detailed K=2 or K=3 tables if present
if 2 in composition_result['k_results']:
    display(composition_result['k_results'][2]['k_set_distribution'])
if 3 in composition_result['k_results']:
    display(composition_result['k_results'][3]['k_set_distribution'])
